# Road Following + VL53L0X — Data collection

Image filename stores **raw mm** (relabel-safe): `<x>_<y>_<tof_left>_<tof_right>_<uuid>.jpg` under `road_following_<A|B>/apex_sensor/`.

### Camera

In [ ]:
# Full reset of the sensors and camera
!echo 'jetson' | sudo -S bash scripts/sensor_soft_reset.sh && printf '\n'
!echo 'jetson' | sudo -S systemctl restart nvargus-daemon && printf '\n'

from jetcam.csi_camera import CSICamera
camera = CSICamera(width=224, height=224, capture_fps=15, flip_method=2)
camera.running = True

### VL53L0X (I2C)

In [ ]:
from robot.vl53l0x import VL53Pair

tof = VL53Pair(bus=1, addr_left=0x28, addr_right=0x29)
print('ToF mm', tof.read_mm())

### Task

In [ ]:
import torchvision.transforms as transforms
from scripts.xy_dataset import XYDataset

TASK = 'road_following'

CATEGORIES = ['apex_sensor']

DATASETS = ['A', 'B']

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}
for name in DATASETS:
    datasets[name] = XYDataset(
        TASK + '_' + name, CATEGORIES, TRANSFORMS,
        random_hflip=True,
        return_sensors=True,  # __getitem__ also returns ToF left/right
    )

### Data Collection

Click the live image to save the apex. The current VL53L0X pair is written into the filename.

ToF left / ToF right follow the sensors live. The strip under the left image and under the live image does too. The strip under the middle image stays on the values saved with that label. The outer edge is 500 mm and stays white when the sensor is farther than that. The fill gets darker toward the center. Grey lines mark 2 car lengths, 1 car length, and the car case. The center red line is 20 mm and is never covered.

In [ ]:
import threading
import cv2
import ipywidgets
import traitlets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jupyter_clickable_image_widget import ClickableImageWidget
from scripts.xy_dataset import tof_clearance_bar, load_bar_thresholds
load_bar_thresholds()


# initialize active dataset
dataset = datasets[DATASETS[0]]

# unobserve all callbacks from camera in case we are running this cell for second time
camera.unobserve_all()

# create image preview
camera_widget = ClickableImageWidget(width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(width=camera.width, height=camera.height)
traitlets.dlink((camera, 'value'), (camera_widget, 'value'), transform=bgr8_to_jpeg)

# create widgets
dataset_widget = ipywidgets.Dropdown(options=DATASETS, description='dataset')
category_widget = ipywidgets.Dropdown(options=dataset.categories, description='category')
count_widget = ipywidgets.IntText(description='count')
tof_left_widget = ipywidgets.IntText(description='ToF left')
tof_right_widget = ipywidgets.IntText(description='ToF right')
BAR_H = 20
def _tof_bar():
    return ipywidgets.Image(
        format='png', width=camera.width, height=BAR_H,
        layout=ipywidgets.Layout(width='%dpx' % camera.width, height='%dpx' % BAR_H,
                                 border='1px solid #bbb', margin='4px 0 0 0'),
    )
tof_bar_widget = _tof_bar()          # live, under the image you click
snapshot_bar_widget = _tof_bar()     # frozen at the saved label
live_bar_widget = _tof_bar()         # live, under the prediction image

# manually update counts at initialization
count_widget.value = dataset.get_count(category_widget.value)

# sets the active dataset
def set_dataset(change):
    global dataset
    dataset = datasets[change['new']]
    count_widget.value = dataset.get_count(category_widget.value)
dataset_widget.observe(set_dataset, names='value')

# update counts when we select a new category
def update_counts(change):
    count_widget.value = dataset.get_count(change['new'])
category_widget.observe(update_counts, names='value')


# ponytail: one I2C reader. Live inference uses the latest pair (one sample behind).
if '_tof_lock' not in globals():
    _tof_lock = threading.Lock()
_tof_latest = (0, 0)
try:
    _tof_poll_stop.set()
except NameError:
    pass
_tof_poll_stop = threading.Event()


def _bar_png(left_mm, right_mm):
    ok, buf = cv2.imencode('.png', tof_clearance_bar(left_mm, right_mm, camera.width, BAR_H))
    return bytes(buf)


def _show_tof(left_mm, right_mm):
    tof_left_widget.value = int(left_mm)
    tof_right_widget.value = int(right_mm)
    png = _bar_png(left_mm, right_mm)
    tof_bar_widget.value = png
    live_bar_widget.value = png


def read_tof_pair():
    global _tof_latest
    with _tof_lock:
        left, right = (int(v) for v in tof.read_mm())
        _tof_latest = (left, right)
        return _tof_latest


def save_snapshot(_, content, msg):
    if content['event'] == 'click':
        data = content['eventData']
        x = data['offsetX']
        y = data['offsetY']

        # read ToF at the moment of the click (raw mm)
        tof_left, tof_right = read_tof_pair()
        _show_tof(tof_left, tof_right)

        # save to disk: <x>_<y>_<tof_left>_<tof_right>_<uuid>.jpg
        dataset.save_entry(category_widget.value, camera.value, x, y,
                           tof_left=tof_left, tof_right=tof_right)

        # display saved snapshot
        snapshot = camera.value.copy()
        snapshot = cv2.circle(snapshot, (x, y), 8, (0, 255, 0), 3)
        snapshot_widget.value = bgr8_to_jpeg(snapshot)
        snapshot_bar_widget.value = _bar_png(tof_left, tof_right)
        count_widget.value = dataset.get_count(category_widget.value)

camera_widget.on_msg(save_snapshot)


def _tof_poll(stop):
    while not stop.is_set():
        try:
            left, right = read_tof_pair()
        except Exception:
            stop.wait(0.3)
            continue

        def _push(left=left, right=right):
            _show_tof(left, right)

        try:
            get_ipython().kernel.io_loop.add_callback(_push)
        except Exception:
            _push()
        stop.wait(0.05)


threading.Thread(target=_tof_poll, args=(_tof_poll_stop,), daemon=True).start()
blank = _bar_png(2000, 2000)  # farther than 2 car lengths: green until a closer reading arrives
tof_bar_widget.value = live_bar_widget.value = snapshot_bar_widget.value = blank

data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([
        ipywidgets.VBox([camera_widget, tof_bar_widget]),
        ipywidgets.VBox([snapshot_widget, snapshot_bar_widget]),
    ], layout=ipywidgets.Layout(align_items='flex-start')),
    dataset_widget,
    category_widget,
    count_widget,
    ipywidgets.HBox([tof_left_widget, tof_right_widget])
])

display(data_collection_widget)

### Model

In [ ]:
import torch
from scripts.resnet_sensor_fusion import create_resnet18_sensor_fusion

device = torch.device('cuda')
output_dim = 2 * len(dataset.categories)  # x, y coordinate for each category

# RESNET 18 + ToF late fusion (see scripts/resnet_sensor_fusion.py)
model = create_resnet18_sensor_fusion(output_dim, pretrained=True)
model = model.to(device)

model_save_button = ipywidgets.Button(description='save model')
model_load_button = ipywidgets.Button(description='load model')
model_path_widget = ipywidgets.Text(description='model path', value='best_steering_model_xy.pth')

def load_model(c):
    ckpt = torch.load(model_path_widget.value)
    state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
    model.load_state_dict(state)
model_load_button.on_click(load_model)

def save_model(c):
    torch.save(model.state_dict(), model_path_widget.value)
model_save_button.on_click(save_model)

model_widget = ipywidgets.VBox([
    model_path_widget,
    ipywidgets.HBox([model_load_button, model_save_button])
])


display(model_widget)

### Live Execution

In [ ]:
import threading
import time
from scripts.utils import preprocess
from scripts.xy_dataset import normalize_tof_tensor

state_widget = ipywidgets.ToggleButtons(options=['stop', 'live'], description='state', value='stop')
prediction_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)

def live(state_widget, model, camera, prediction_widget):
    global dataset
    while state_widget.value == 'live':
        image = camera.value
        preprocessed = preprocess(image)

        # numbers and the strip are already live; model gets 0-500 mm clamped, then 1->0
        tof_left, tof_right = _tof_latest
        sensors = normalize_tof_tensor(
            torch.tensor([[tof_left, tof_right]], dtype=torch.float32)
        ).to(device)

        # fusion forward: image features concat ToF embedding, then (x, y)
        output = model(preprocessed, sensors).detach().cpu().numpy().flatten()
        category_index = dataset.categories.index(category_widget.value)
        x = output[2 * category_index]
        y = output[2 * category_index + 1]

        x = int(camera.width * (x / 2.0 + 0.5))
        y = int(camera.height * (y / 2.0 + 0.5))

        prediction = image.copy()
        prediction = cv2.circle(prediction, (x, y), 8, (255, 0, 0), 3)
        prediction_widget.value = bgr8_to_jpeg(prediction)

def start_live(change):
    if change['new'] == 'live':
        execute_thread = threading.Thread(target=live, args=(state_widget, model, camera, prediction_widget))
        execute_thread.start()

state_widget.observe(start_live, names='value')

live_execution_widget = ipywidgets.VBox([
    ipywidgets.VBox([prediction_widget, live_bar_widget]),
    state_widget
])

display(live_execution_widget)

### Train

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
from scripts.xy_dataset import load_bar_thresholds, save_bar_thresholds

_th = load_bar_thresholds()
_field = {"description_width": "initial"}
two_car_widget = ipywidgets.IntText(description="2 car lengths", value=_th["two_car"], style=_field, layout=ipywidgets.Layout(width="200px"))
one_car_widget = ipywidgets.IntText(description="1 car length", value=_th["one_car"], style=_field, layout=ipywidgets.Layout(width="190px"))
case_widget = ipywidgets.IntText(description="car case", value=_th["car_case"], style=_field, layout=ipywidgets.Layout(width="160px"))
bar_apply_button = ipywidgets.Button(description="apply")
bar_apply_status = ipywidgets.HTML("")

def _apply_bar(_):
    try:
        save_bar_thresholds(two_car_widget.value, one_car_widget.value, case_widget.value)
        bar_apply_status.value = "saved"
    except ValueError as e:
        bar_apply_status.value = str(e)

bar_apply_button.on_click(_apply_bar)
bar_threshold_widget = ipywidgets.HBox([
    two_car_widget, one_car_widget, case_widget, bar_apply_button, bar_apply_status
])

BATCH_SIZE = 8

optimizer = torch.optim.Adam(model.parameters())
# optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

epochs_widget = ipywidgets.IntText(description='epochs', value=1)
eval_button = ipywidgets.Button(description='evaluate')
train_button = ipywidgets.Button(description='train')
loss_widget = ipywidgets.FloatText(description='loss')
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

loss_epoch_values = []
loss_values = []
loss_plot_output = ipywidgets.Output()

def _epoch_tick_step(max_epoch):
    """Pick a clean integer step so x labels stay readable."""
    if max_epoch <= 8:
        return 1
    best = 1
    for step in (2, 5, 10, 20, 25, 50):
        count = max_epoch / step
        if 3 <= count <= 8:
            best = step
    if best > 1:
        return best
    for step in (2, 5, 10, 20, 25, 50):
        if max_epoch / step <= 8:
            return step
    return max(50, (max_epoch + 7) // 8)

def update_loss_plot():
    with loss_plot_output:
        clear_output(wait=True)
        if not loss_values:
            return
        max_epoch = max(loss_epoch_values)
        step = _epoch_tick_step(max_epoch)
        xticks = list(range(step, max_epoch + 1, step))
        if xticks[-1] != max_epoch:
            xticks.append(max_epoch)
        plt.figure(figsize=(5, 3))
        plt.plot(loss_epoch_values, loss_values, marker='o')
        plt.xticks(xticks)
        plt.xlim(0.5, max_epoch + 0.5)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Live Training Loss')
        plt.grid(True)
        plt.show()

def train_eval(is_training):
    global BATCH_SIZE, model, dataset, optimizer, eval_button, train_button, loss_widget, progress_widget, state_widget

    try:
        train_loader = torch.utils.data.DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        state_widget.value = 'stop'
        train_button.disabled = True
        eval_button.disabled = True
        time.sleep(1)

        best_loss = float('inf')
        best_epoch = 0
        best_state = None

        if is_training:
            model = model.train()
            current_epoch = 0
            loss_epoch_values.clear()
            loss_values.clear()
            update_loss_plot()
        else:
            model = model.eval()

        while epochs_widget.value > 0:
            i = 0
            sum_loss = 0.0
            for images, category_idx, xy, sensors in iter(train_loader):
                # send data to device
                images = images.to(device)
                xy = xy.to(device)
                sensors = sensors.to(device)  # 0-500 mm clamped, then 1->0

                if is_training:
                    # zero gradients of parameters
                    optimizer.zero_grad()

                # execute model to get outputs (image + ToF fusion)
                outputs = model(images, sensors)

                # compute MSE loss over x, y coordinates for associated categories
                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx+2] - xy[batch_idx])**2)
                loss /= len(category_idx)

                if is_training:
                    # run backpropogation to accumulate gradients
                    loss.backward()

                    # step optimizer to adjust parameters
                    optimizer.step()

                # increment progress
                count = len(category_idx.flatten())
                i += count
                sum_loss += float(loss)
                progress_widget.value = i / len(dataset)
                loss_widget.value = sum_loss / i

            if is_training:
                # One plot point per finished epoch: x = 1, 2, 3, ...
                current_epoch += 1
                epoch_loss = loss_widget.value
                loss_epoch_values.append(current_epoch)
                loss_values.append(epoch_loss)
                update_loss_plot()

                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_epoch = current_epoch
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    torch.save(best_state, model_path_widget.value)

                epochs_widget.value = epochs_widget.value - 1
            else:
                break

        if is_training and best_state is not None:
            model.load_state_dict(best_state)
            print(f'Best model restored: epoch {best_epoch}, loss {best_loss:.6f} -> {model_path_widget.value}')
    except Exception as e:
        print('Training error:', e)
    model = model.eval()

    train_button.disabled = False
    eval_button.disabled = False
    state_widget.value = 'live'

train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))

train_eval_widget = ipywidgets.VBox([
    bar_threshold_widget,
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([train_button, eval_button])
])

display(train_eval_widget)

### All together!

The following widget can be used to label a multi-class x, y dataset.  It supports labeling only one instance of each class per image (ie: only one dog), but multiple classes (ie: dog, cat, horse) per image are possible.

Click the image on the top left to save an image of ``category`` to ``dataset`` at the clicked location. The current ToF readings are stored in the filename. Strips under the left and right images follow the sensors live. The strip under the middle image is the pair saved with that label. The outer edge is 500 mm. Each side is grey and gets darker as that sensor reads closer. Grey lines mark 2 car lengths, 1 car length, and the car case. The center red line is 20 mm.

| Widget | Description |
|--------|-------------|
| dataset | Selects the active dataset |
| category | Selects the active category |
| 2 car lengths / 1 car length / car case | Distances for the colored lines. Apply saves them in `scripts/bar_thresholds.json` |
| apply | Updates the lines and the fill colors, and keeps the values after a reboot |
| epochs | Sets the number of epochs to train for |
| train | Trains on the active dataset for the number of epochs specified |
| evaluate | Evaluates the accuracy on the active dataset over one epoch |
| model path | Sets the active model path |
| load | Loads a model from the active model path |
| save | Saves a model to the active model path |
| stop | Disables the live demo |
| live | Enables the live demo |
| reset sensors | Fixes the sensor addresses, then resets them, without restarting this page |

In [ ]:
import threading

reset_sensors_button = ipywidgets.Button(description="reset sensors")

def _on_reset_sensors(_):
    reset_sensors_button.disabled = True

    def work():
        try:
            tof.request_reset(blocking=True)
        finally:
            def enable():
                reset_sensors_button.disabled = False
            try:
                get_ipython().kernel.io_loop.add_callback(enable)
            except Exception:
                enable()

    threading.Thread(target=work, daemon=True).start()

reset_sensors_button.on_click(_on_reset_sensors)

all_widget = ipywidgets.VBox([
    ipywidgets.HBox([data_collection_widget, live_execution_widget]),
    train_eval_widget,
    model_widget,
    loss_plot_output,
    reset_sensors_button,
])

display(all_widget)

After collecting, zip `apex_sensor` for training elsewhere, or use `scripts/train_model.ipynb`.

In [ ]:
!cd road_following_A && zip -r - apex_sensor > ../dataset_sensor.zip

In [ ]:
camera.running = False
camera.unobserve_all()
camera.cap.release()